Iterate through all `*_privacy_candidates_kept.json` files in the directory.

Parse the following from each filename:

- `apkname`, for example `ae.brandsforless.android`
- `version_code`, for example `412`

Then find the corresponding `*-412_urls.json` file.

Read each URL from the `privacy_candidates_kept.json` file and match it against the same URL in the corresponding `urls.json` file.

Generate one JSON file per APK.

In [1]:
import os
import re
import json
from pathlib import Path
from typing import Dict, List, Any
from collections import defaultdict

In [ ]:

# Whether to traverse subdirectories recursively
RECURSIVE = True


def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def iter_files(root: Path):
    if RECURSIVE:
        yield from root.rglob("*")
    else:
        yield from root.iterdir()


def parse_kept_filename(filename: str):
    """
    Example:
      ae.brandsforless.android-412_privacy_candidates_kept.json
    Parsed result:
      apkname = ae.brandsforless.android
      version_code = 412
    """
    m = re.match(r"^(?P<apkname>.+)-(?P<ver>\d+)_privacy_candidates_kept\.(json|josn)$", filename)
    if not m:
        return None
    return m.group("apkname"), m.group("ver")

def find_matching_urls_file(kept_path: Path, apkname: str, version_code: str):
    for suffix in ["json", "josn"]:
        p = kept_path.with_name(f"{apkname}-{version_code}_urls.{suffix}")
        if p.exists():
            return p
    return None

def build_url_index(urls_data: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    """
    Build an index from the `urls` list in the `*_urls.json` file:
      { url_string: url_entry_dict }
    """
    result = {}
    for item in urls_data.get("urls", []):
        u = item.get("url")
        if u:
            result[u] = item
    return result

In [ ]:
### first batch ###
# [
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA1_first_328_batch
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA2_second_100_batch
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA3_third_100_batch
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA4_forth_100_batch
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA5_fifth_100_batch
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA6_sisth_100_batch
# ]

# [
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA1_first_328_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA2_second_100_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA3_third_100_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA4_forth_100_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA5_fifth_100_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA6_sisth_74_batch
# ]

In [ ]:
# =========================
# Configuration
# =========================
ROOT_DIR = r"1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA6_sisth_100_batch"
OUT_DIR = Path(r"1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA6_sisth_74_batch")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def main():
    root = Path(ROOT_DIR)

    count = 1
    # Optional: record exceptions
    missing_pairs = []
    parse_failed_files = []

    for p in iter_files(root):
        # Each APK corresponds to a list
        apk_outputs: Dict[str, List[Dict[str, Any]]] = defaultdict(list)
        if not p.is_file():
            continue
        if not re.match(r"^.+-\d+_privacy_candidates_kept\.(json|josn)$", p.name):
            continue

        parsed = parse_kept_filename(p.name)
        if not parsed:
            parse_failed_files.append({
                "file": str(p),
                "error": "filename parse failed"
            })
            continue

        apkname, version_code = parsed
        count = count + 1

        urls_file = find_matching_urls_file(p, apkname, version_code)
        if urls_file is None:
            missing_pairs.append({
                "apkname": apkname,
                "version_code": version_code,
                "kept_file": str(p),
                "expected_urls_file": f"{apkname}-{version_code}_urls.json"
            })
            continue

        try:
            kept_data = load_json(p)
        except Exception as e:
            parse_failed_files.append({
                "file": str(p),
                "error": f"read kept json failed: {e}"
            })
            continue

        try:
            urls_data = load_json(urls_file)
        except Exception as e:
            parse_failed_files.append({
                "file": str(urls_file),
                "error": f"read urls json failed: {e}"
            })
            continue

        if not isinstance(kept_data, list):
            parse_failed_files.append({
                "file": str(p),
                "error": "kept json is not a list"
            })
            continue

        url_index = build_url_index(urls_data)

        for candidate in kept_data:
            url = candidate.get("url")
            print(url)
            if not url:
                continue

            record = {
                "apkname": apkname,
                "version_code": version_code,
                "url": url,
                "privacy_candidate_info": candidate,
                "url_evidence_info": url_index.get(url)
            }

            apk_outputs[apkname].append(record)
            out_file = OUT_DIR / f"{apkname}_{version_code}_merged_privacy_url_evidence.json"
            save_json(apk_outputs, out_file)

    print(f"done, total files processed: {count}")


if __name__ == "__main__":
    main()

https://dash.applovin.com/documentation/mediation/android/getting-started/terms-and-privacy-policy-flow
https://dash.applovin.com/documentation/mediation/android/getting-started/terms-and-privacy-policy-flow#enabling-google-ump
https://lutech.vn/privacy
https://www.applovin.com/privacy/
https://sumeria.eu/en/essentials/terms-and-conditions/lydia-solutions-privacy-policy/?template=hidden
https://sumeria.eu/en/essentials/terms-and-conditions/lydia-solutions-privacy-policy?hide=true&amp;hide_chat=true&amp;hideQuestion=true
https://www.paylead.fr/privacy
https://sumeria.eu/en/essentials/terms-and-conditions/lydia-solutions-privacy-policy/?template=hidden
https://sumeria.eu/en/essentials/terms-and-conditions/lydia-solutions-privacy-policy?hide=true&amp;hide_chat=true&amp;hideQuestion=true
https://www.paylead.fr/privacy
https://developers.applovin.com/en/android/overview/terms-and-privacy-policy-flow
https://developers.applovin.com/en/android/overview/terms-and-privacy-policy-flow#enabling-g

https://suji.games/privacy.html
https://vungle.com/privacy/
https://www.applovin.com/privacy/
https://www.smaato.com/privacy/
https://www.termsfeed.com/privacy-policy/0f94ae664356e78a72468543245251b4huhubhyhz-gb-2312ii1i2i4i8iAdFrameworkEnablediOSiap
https://developers.applovin.com/en/android/overview/terms-and-privacy-policy-flow
https://developers.applovin.com/en/android/overview/terms-and-privacy-policy-flow#enabling-google-ump
https://developers.applovin.com/en/unity/overview/terms-and-privacy-policy-flow
https://suji.games/privacy.html
https://www.applovin.com/privacy/
https://www.smaato.com/privacy/
https://dash.applovin.com/documentation/mediation/android/getting-started/terms-and-privacy-policy-flow
https://dash.applovin.com/documentation/mediation/android/getting-started/terms-and-privacy-policy-flow#enabling-google-ump
https://theholycowstudio.com/privacy-policy/
https://www.applovin.com/privacy/
https://dash.applovin.com/documentation/mediation/android/getting-started/terms-